# Baseline model - PaDiM on AutoVI

Stage 1 baseline, centralised training. The model is PaDiM (Defard et al., 2021), one of the six methods
benchmarked on AutoVI by Carvalho et al. (2024). It is unsupervised: it is fitted on normal images only
and scores an image by how far its patches are from what it saw.

Why this one out of the six: it needs a single pass over the data (no gradient training), it runs on CPU,
and what it learns is a mean and a covariance per patch position. Those are averages, so the same model
can be aggregated over clients in the federated part.

The notebook
1. loads the client partitions from `data/partitions/autovi`
2. defines the model
3. fits it on the pooled data of all clients (centralised) and on each client alone (local), seeds 1, 2, 3
4. reports the results as mean ± std over the seeds

On fep run it as a job, not on the login node.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
                             f1_score, roc_auc_score, roc_curve)
from torchvision.models import ResNet18_Weights, resnet18

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_splitting").is_dir())
PARTS = REPO / "data" / "partitions" / "autovi"
RUNS = REPO / "outputs" / "experiments" / "padim-central"
FIGS = REPO / "outputs" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

SEEDS = [1, 2, 3]
N_DIMS = 100    # feature channels kept, out of 448
BATCH = 32
TNR = 0.95      # threshold = this quantile of the normal validation scores
SIGMA = 4       # smoothing of the anomaly map, for a 224 px image

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("repo  :", REPO)
print("data  :", PARTS)
print("device:", dev)

### Data

The partitions are the ones written by `data_splitting/split_data.py`, one client per production part.
Labels: 0 = normal, 1 = anomaly.

Training uses the normal images only. Anomalous images that are in a client's training split are left
out, and the labels of the validation and test images are used for evaluation only.

In [ ]:
def read(path, split):
    d = np.load(path)
    return d[f"x_{split}"], d[f"y_{split}"].reshape(-1)


files = sorted(PARTS.glob("client_*.npz"), key=lambda p: int(p.stem.split("_")[1]))
assert files, f"no partitions in {PARTS}, run data_splitting/split_data.py first"

train = [read(f, "train") for f in files]
val = [read(f, "val") for f in files]
x_test, y_test = read(PARTS / "test.npz", "test")

counts = pd.DataFrame(
    [[(ytr == 0).sum(), (ytr == 1).sum(), (yva == 0).sum(), (yva == 1).sum()]
     for (_, ytr), (_, yva) in zip(train, val)],
    columns=["train normal", "train anomaly (unused)", "val normal", "val anomaly"])
counts.index.name = "client"

print("images:", x_test.shape[1:], x_test.dtype)
print(f"test  : {(y_test == 0).sum()} normal, {(y_test == 1).sum()} anomaly")
counts

In [ ]:
x_pool = np.concatenate([x[y == 0] for x, y in train])
x_val = np.concatenate([x for x, _ in val])
y_val = np.concatenate([y for _, y in val])
owner = np.concatenate([np.full(len(y), c) for c, (_, y) in enumerate(val)])

print("pooled normal training images:", len(x_pool))
print("pooled validation images     :", len(x_val))

### Model

**Features.** Every image goes through a ResNet-18 pretrained on ImageNet, which stays frozen. The outputs
of `layer1`, `layer2` and `layer3` are brought to the resolution of `layer1` (a quarter of the image) and
concatenated: 64 + 128 + 256 = 448 channels for every patch position. `N_DIMS` of these channels are kept,
picked at random. The seed decides which ones, and that is the only random part of the model.

**Fit.** For every patch position the model estimates a Gaussian over the normal training images, with
mean $\mu_p$ and covariance $\Sigma_p$ (plus $0.01\,I$ so it can be inverted). `add` goes through the
images and only keeps their number, $\sum x$ and $\sum x x^\top$ per position; `finish` turns these
sums into the mean and the inverse covariance.

**Score.** The anomaly map is the Mahalanobis distance of every patch to the Gaussian of its position,
resized to the image and smoothed. The image score is the maximum of the map.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


class Backbone(torch.nn.Module):
    def __init__(self):
        super().__init__()
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.stem = torch.nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)
        self.l1, self.l2, self.l3 = r.layer1, r.layer2, r.layer3

    def forward(self, x):
        f1 = self.l1(self.stem(x))
        f2 = self.l2(f1)
        f3 = self.l3(f2)
        hw = f1.shape[-2:]
        f2 = F.interpolate(f2, size=hw, mode="nearest")
        f3 = F.interpolate(f3, size=hw, mode="nearest")
        return torch.cat([f1, f2, f3], dim=1)


net = Backbone().to(dev).eval()


def batches(x):
    for i in range(0, len(x), BATCH):
        b = torch.from_numpy(x[i:i + BATCH]).to(dev)
        b = b.permute(0, 3, 1, 2).float() / 255
        yield (b - MEAN) / STD

In [ ]:
class Padim:
    def __init__(self, seed):
        g = torch.Generator().manual_seed(seed)
        self.idx = torch.randperm(448, generator=g)[:N_DIMS].to(dev)
        self.n = 0

    @torch.no_grad()
    def embed(self, xb):
        f = net(xb)[:, self.idx]
        self.hw = tuple(f.shape[-2:])
        return f.flatten(2).transpose(1, 2)     # (batch, positions, dims)

    def add(self, x):
        for xb in batches(x):
            e = self.embed(xb).double().transpose(0, 1)     # (positions, batch, dims)
            if self.n == 0:
                p, _, d = e.shape
                self.s1 = e.new_zeros(p, d)         # sum of x
                self.s2 = e.new_zeros(p, d, d)      # sum of outer products
            self.s1 += e.sum(1)
            self.s2.baddbmm_(e.transpose(1, 2), e)
            self.n += e.shape[1]
        return self

    def finish(self):
        mu = self.s1 / self.n
        cov = self.s2 / (self.n - 1)
        cov -= self.n / (self.n - 1) * mu[:, :, None] * mu[:, None, :]
        cov += 0.01 * torch.eye(cov.shape[-1], dtype=cov.dtype, device=cov.device)
        self.mu = mu.float()
        # inverted in chunks, all positions at once takes too much memory on large images
        self.icov = torch.cat([torch.linalg.inv(c).float() for c in cov.split(1024)])
        return self

    @torch.no_grad()
    def maps(self, x):
        out = []
        for xb in batches(x):
            d = self.embed(xb) - self.mu
            m = torch.einsum("bpi,pij,bpj->bp", d, self.icov, d).clamp(min=0).sqrt()
            m = F.interpolate(m.view(-1, 1, *self.hw), size=xb.shape[-2:],
                              mode="bilinear", align_corners=False)
            out.append(m[:, 0].cpu().numpy())
        out = np.concatenate(out)
        s = SIGMA * out.shape[-1] / 224
        return gaussian_filter(out, sigma=(0, s, s))

    def score(self, x):
        return self.maps(x).reshape(len(x), -1).max(1)

### Evaluation

AUROC and average precision do not need a threshold; they are the image-level metrics of the paper.

For a normal / anomaly decision a threshold is needed. It is set to the 95th percentile of the scores of
the **normal validation images**, so roughly 5% false alarms, without looking at any defect. Balanced
accuracy, macro-F1, TPR and TNR are computed at that threshold.

The centralised model has one threshold for all clients. A local model has its own.

In [ ]:
def evaluate(y, s, thr):
    pred = (s > thr).astype(int)
    both = len(np.unique(y)) == 2
    return {
        "accuracy": float((pred == y).mean()),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "f1_macro": float(f1_score(y, pred, average="macro", zero_division=0)),
        "auroc": float(roc_auc_score(y, s)) if both else float("nan"),
        "ap": float(average_precision_score(y, s)) if both else float("nan"),
        "tpr": float(pred[y == 1].mean()) if both else float("nan"),
        "tnr": float(1 - pred[y == 0].mean()),
    }

### Centralised and local runs

Centralised: one model fitted on the normal training images of all clients together. This is the
counterpart of the federated model.

Local: one model per client, fitted on that client's normal images only and evaluated on its own
validation set. The paper also trains one model per part, so this is the run to compare with its numbers.

Every seed is saved in `outputs/experiments/padim-central/seed<k>/`, with a `results.json` in the format
of `experiments/evaluation.py`, so `experiments/summarize.py` reads it like any other run.

In [ ]:
results, local = {}, {}

for seed in SEEDS:
    central = Padim(seed).add(x_pool).finish()
    s_val, s_test = central.score(x_val), central.score(x_test)
    thr = float(np.quantile(s_val[y_val == 0], TNR))

    res = {"val": evaluate(y_val, s_val, thr), "val_per_client": {},
           "test": evaluate(y_test, s_test, thr), "threshold": thr}
    for c in range(len(files)):
        m = owner == c
        res["val_per_client"][str(c)] = {"samples": int(m.sum()),
                                         **evaluate(y_val[m], s_val[m], thr)}

    loc = {}
    for c, ((xtr, ytr), (xva, yva)) in enumerate(zip(train, val)):
        s = Padim(seed).add(xtr[ytr == 0]).finish().score(xva)
        loc[str(c)] = evaluate(yva, s, float(np.quantile(s[yva == 0], TNR)))

    out = RUNS / f"seed{seed}"
    out.mkdir(parents=True, exist_ok=True)
    torch.save({"idx": central.idx.cpu(), "mean": central.mu.cpu(), "icov": central.icov.cpu(),
                "n": central.n, "hw": central.hw}, out / "final_model.pt")
    cfg = {"partitions-dir": str(PARTS), "seed": seed, "mode": "centralised",
           "model": "padim-resnet18", "n-dims": N_DIMS, "tnr": TNR}
    (out / "metrics.json").write_text(json.dumps({"run_config": cfg}, indent=2))
    (out / "results.json").write_text(json.dumps(res, indent=2))
    (out / "local.json").write_text(json.dumps(loc, indent=2))

    results[seed], local[seed] = res, loc
    print(f"seed {seed}: val auroc {res['val']['auroc']:.3f}, "
          f"val balanced acc {res['val']['balanced_accuracy']:.3f}")

### Results

Mean ± std over the seeds (std with ddof = 1, as in `experiments/summarize.py`). The seeds only change
which feature channels are kept.

The validation columns are the ones to compare models on. The thresholded validation numbers are a bit
optimistic because the threshold was set on the same normal images; AUROC and AP are not affected. The
test column is the number for the report.

In [ ]:
def mean_std(v):
    v = np.array(v, dtype=float)
    return f"{v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}"


keys = ["auroc", "ap", "balanced_accuracy", "f1_macro", "accuracy", "tpr", "tnr"]
summary = pd.DataFrame(
    {split: [mean_std([results[s][split][k] for s in SEEDS]) for k in keys]
     for split in ("val", "test")}, index=keys)
summary

Per client, on its own validation set: the centralised model against the model fitted on that client
alone. With the group split the clients are ordered by size, so client 0 should be `engine_wiring`,
1 `pipe_clip` and 2 `pipe_staple` (check the train sizes in `meta.json`). The validation sets are small,
so differences of a few points between clients are not reliable.

In [ ]:
rows = []
for c in map(str, range(len(files))):
    row = {"client": c, "val images": results[SEEDS[0]]["val_per_client"][c]["samples"]}
    for k, name in (("auroc", "auroc"), ("ap", "ap"), ("balanced_accuracy", "bal. acc.")):
        row[f"central {name}"] = mean_std([results[s]["val_per_client"][c][k] for s in SEEDS])
        row[f"local {name}"] = mean_std([local[s][c][k] for s in SEEDS])
    rows.append(row)
per_client = pd.DataFrame(rows).set_index("client")
per_client

For reference, PaDiM in Carvalho et al. (2024), Tables 6 and 7, image level, in %:

| part | AUROC | AP |
|---|---|---|
| engine_wiring | 79.2 ± 1.3 | 77.2 ± 1.3 |
| pipe_clip | 62.6 ± 2.0 | 48.5 ± 1.6 |
| pipe_staple | 54.6 ± 3.0 | 39.4 ± 1.2 |

This is only a rough check for the local models. The paper uses the official AutoVI split (all defects in
the test set) and 256 px images; here the images were pooled, deduplicated and split again, and the size
is the one stored in the partitions.

In [ ]:
# figures use the model of the last seed, still in memory
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bins = np.linspace(s_test.min(), s_test.max(), 40)
ax[0].hist(s_test[y_test == 0], bins=bins, alpha=0.6, label="normal")
ax[0].hist(s_test[y_test == 1], bins=bins, alpha=0.6, label="anomaly")
ax[0].axvline(thr, color="k", ls="--", lw=1, label="threshold")
ax[0].set_xlabel("image score")
ax[0].set_ylabel("test images")
ax[0].legend()

fpr, tpr, _ = roc_curve(y_test, s_test)
ax[1].plot(fpr, tpr, label=f"AUROC {roc_auc_score(y_test, s_test):.3f}")
ax[1].plot([0, 1], [0, 1], color="grey", ls=":", lw=1)
ax[1].set_xlabel("false positive rate")
ax[1].set_ylabel("true positive rate")
ax[1].legend()

fig.suptitle(f"centralised PaDiM, test set, seed {seed}")
fig.tight_layout()
fig.savefig(FIGS / "baseline_scores.png", dpi=150)

In [ ]:
rng = np.random.default_rng(0)
pick = np.concatenate([rng.choice(np.where(y_test == k)[0], 4, replace=False) for k in (0, 1)])
amaps = central.maps(x_test[pick])

fig, ax = plt.subplots(2, len(pick), figsize=(2 * len(pick), 4.4))
for j, i in enumerate(pick):
    ax[0, j].imshow(x_test[i])
    ax[0, j].set_title(f"{'anomaly' if y_test[i] else 'normal'}\nscore {s_test[i]:.1f}", fontsize=8)
    ax[1, j].imshow(x_test[i])
    ax[1, j].imshow(amaps[j], cmap="jet", alpha=0.5, vmin=amaps.min(), vmax=amaps.max())
for a in ax.ravel():
    a.axis("off")
fig.suptitle("AutoVI © Renault Group - CC BY-NC-SA 4.0", fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / "baseline_maps.png", dpi=150)

### Client statistics add up

The model is fully described by the number of images, $\sum x$ and $\sum x x^\top$. Adding these over
the local models gives the centralised model back (up to floating point). This is what the
federated version relies on: a weighted average of the clients' means and second moments, which is what
FedAvg computes, is the mean and second moment of the pooled data.

In [ ]:
n, s1, s2 = 0, 0, 0
for x, y in train:
    m = Padim(seed).add(x[y == 0])
    n += m.n
    s1 += m.s1
    s2 += m.s2
    del m

print("images     :", n, "vs", central.n)
print("sum x      :", float((s1 - central.s1).abs().max() / central.s1.abs().max()))
print("sum x x^T  :", float((s2 - central.s2).abs().max() / central.s2.abs().max()))

### Observations

*(to be written after the run on the real data)*

### Limitations

- Not the official AutoVI protocol: the partitions come from a pooled, deduplicated and re-split version
  of the dataset, so the numbers are not directly comparable with the published ones.
- Image level only. The partitions do not carry the ground-truth masks, so the anomaly maps are not
  scored (no AUsPRO).
- `test.npz` has no client id, so results per client exist for the validation sets only.
- One threshold for three different parts in the centralised model.

### References

Carvalho, P., Lafou, M., Durupt, A., Leblanc, A., & Grandvalet, Y. (2024). Detecting visual anomalies in
an industrial environment: Unsupervised methods put to the test on the AutoVI dataset. *Computers in
Industry, 163*, 104151. https://doi.org/10.1016/j.compind.2024.104151

Defard, T., Setkov, A., Loesch, A., & Audigier, R. (2021). PaDiM: A patch distribution modeling framework
for anomaly detection and localization. *ICPR 2020 Workshops*. https://arxiv.org/abs/2011.08785

He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*.